# 01-基础对话 - Kimi API

文档: https://platform.moonshot.ai/docs/

## 环境准备

在项目根目录创建 `.env` 文件，添加：
```
VITE_KIMI_API_KEY=your-api-key
```

In [17]:
# 安装依赖
# !pip install openai python-dotenv

In [18]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# 加载 .env 文件
load_dotenv(dotenv_path='../../.env')  # 根据实际路径调整
# 从环境变量获取 API Key
api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
if not api_key:
    raise ValueError("❌ 未找到 API Key，请在 .env 文件中设置 VITE_KIMI_API_KEY")
print("✅ API Key 加载成功, KIMI_API_KEY = ", api_key)
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.cn/v1")

# 初始化客户端
client = OpenAI(
    api_key=api_key,
    base_url=base_url,
)

print("✅ Kimi 客户端初始化成功")
print(f"📍 Base URL: {base_url}")

✅ API Key 加载成功, KIMI_API_KEY =  sk-tkxFpdhGtYivXgurCSWf9mRpBQwovUuwcLHHOrG29wFv7WFm
✅ Kimi 客户端初始化成功
📍 Base URL: https://api.moonshot.cn/v1


## 基础调用（非流式）

In [ ]:
# 基础对话 - kimi-k2-turbo-preview
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[
        {"role": "system", "content": "You are Kimi, an AI assistant provided by Moonshot AI."},
        {"role": "user", "content": "Hello, what is 1+1?"}
    ],
    temperature=1,
)

print(f"回复: {response.choices[0].message.content}")

回复: Hello! 1 + 1 equals **2**.


## 使用不同模型

In [20]:
# 测试不同模型
models = ["kimi-k2.5", "kimi-k2-thinking"]

for model in models:
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Hello!"}],
        )
        print(f"{model}:")
        print(response.choices[0].message.content)
        print()
    except Exception as e:
        print(f"❌ {model} 错误: {e}")
        print()

kimi-k2.5:
Hello! How can I help you today?

kimi-k2-thinking:
Hello! How can I help you today?



## 带错误处理的调用

In [21]:
from openai import APIError, APIConnectionError

def safe_chat(message: str, model: str = "kimi-k2-turbo-preview") -> str:
    """带错误处理的对话函数"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": message}
            ],
        )
        return response.choices[0].message.content
    except APIConnectionError as e:
        return f"连接错误: {e}"
    except APIError as e:
        return f"API 错误: {e}"
    except Exception as e:
        return f"未知错误: {e}"

# 使用
result = safe_chat("Hello")
print(f"✅ 调用成功")
print(f"回复: {result}")

✅ 调用成功
回复: Hi there! How can I help you today?


## 响应结构解析

In [22]:
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[{"role": "user", "content": "Hello"}],
)

print("响应结构:")
print(f"ID: {response.id}")
print(f"Model: {response.model}")
print()

print("消息内容:")
message = response.choices[0].message
print(f"Role: {message.role}")
print(f"Content: {message.content}")
print()

print("Token 使用:")
usage = response.usage
print(f"Prompt tokens: {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")

响应结构:
ID: chatcmpl-69a32ee5ec6aff7baf5de020
Model: kimi-k2-turbo-preview

消息内容:
Role: assistant
Content: Hello! How can I help you today?

Token 使用:
Prompt tokens: 8
Completion tokens: 10
Total tokens: 18
